# Anexo II · Laboratório: emendando os vencimentos do contrato futuro
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**: oito contratos de petróleo WTI (Nymex), cinco de açúcar nº 11 (ICE, só na Parte 6, exemplo extra do hub), o Ibovespa desde 2005 e o CDI diário do Banco Central (série SGS 12). Na primeira execução, o notebook baixa os dados do Yahoo Finance e do Banco Central na sua máquina e guarda uma cópia local (`dados_anexo2_perpetuo.pkl` e, na Parte 6, `dados_anexo2_acucar.pkl`), que as execuções seguintes reaproveitam. O Yahoo corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. A troca no petróleo é didática; o minicontrato de Ibovespa é calculado pelo preço justo. Testes didáticos, sem custo: **não são recomendação**.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import requests
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
CONTRATOS = ["CLZ26.NYM", "CLF27.NYM", "CLJ27.NYM", "CLN27.NYM", "CLU27.NYM", "CLZ27.NYM", "CLH28.NYM", "CLM28.NYM"]
INI, FIM = "2025-01-01", "2026-10-01"
CACHE = "dados_anexo2_perpetuo.pkl"


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")


def cdi_bcb(ini=2005, fim=2026):
    """CDI diário (% ao dia) do Banco Central, série SGS 12, em blocos de até 10 anos."""
    partes = []
    for a in range(ini, fim + 1, 10):
        url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.12/dados?formato=json"
               f"&dataInicial=01/01/{a}&dataFinal=31/12/{min(a + 9, fim)}")
        for tentativa in range(5):
            try:
                js = requests.get(url, timeout=60).json()
                break
            except Exception:
                time.sleep(5 * (tentativa + 1))
        else:
            raise RuntimeError("Não consegui baixar o CDI do Banco Central depois de 5 tentativas. O site pode estar"
                               " fora do ar. Espere alguns minutos e rode a célula de novo.")
        partes.append(pd.Series({pd.to_datetime(x["data"], dayfirst=True): float(x["valor"]) / 100 for x in js}))
    return pd.concat(partes).sort_index()

## Parte 1: os contratos de petróleo e o degrau de cada troca

In [ ]:
if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    px, ibov, cdi = pd.read_pickle(CACHE)
else:
    px = pd.concat({c: baixa(c, start=INI, end=FIM, auto_adjust=False, multi_level_index=False)["Close"]
                    for c in CONTRATOS}, axis=1, sort=True)
    ibov = baixa("^BVSP", start="2005-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    cdi = cdi_bcb()
    pd.to_pickle((px, ibov, cdi), CACHE)
px = px.dropna()
pc = lambda x, k=1: f"{x:.{k}%}"

n = len(px)
trocas = [px.index[int(n * k / len(CONTRATOS))] for k in range(1, len(CONTRATOS))]   # 7 dias de troca
vigente = pd.Series(CONTRATOS[0], index=px.index)
for k, dt in enumerate(trocas, start=1):
    vigente[px.index >= dt] = CONTRATOS[k]


def emenda(precos, vig, dias):
    """Devolve as séries sem ajuste, por diferença e por razão. precos: um contrato por coluna; vig: contrato do dia."""
    sem = pd.Series([precos.at[d, c] for d, c in vig.items()], index=precos.index)
    dif, raz, saltos = sem.copy(), sem.copy(), []
    for dt in dias:
        k = precos.index.get_loc(dt)
        velho, novo = vig.iloc[k - 1], vig.iloc[k]
        gap = precos.at[dt, novo] - precos.at[dt, velho]
        fator = precos.at[dt, novo] / precos.at[dt, velho]
        antes = precos.index < dt
        dif[antes] += gap                      # o passado anda em pontos
        raz[antes] *= fator                    # o passado anda em porcentagem
        saltos.append((dt, velho, novo, gap, fator - 1))
    return sem, dif, raz, saltos


def carteira(precos, vig):
    """O que ganhou quem ficou com 1 contrato e rolou: variação diária do contrato que tinha no dia anterior."""
    ant = vig.shift(1).iloc[1:]
    pts = pd.Series([precos.at[d, c] - precos.at[p, c] for p, (d, c) in zip(precos.index[:-1], ant.items())],
                    index=precos.index[1:])
    ret = pd.Series([precos.at[d, c] / precos.at[p, c] - 1 for p, (d, c) in zip(precos.index[:-1], ant.items())],
                    index=precos.index[1:])
    return pts, ret


sem, dif, raz, saltos = emenda(px, vigente, trocas)
mostra("Parte 1. Petróleo WTI: preço do contrato novo menos o do velho no dia da troca")
for dt, v, nv, g, f in saltos:
    mostra(f"   {dt:%d/%m/%Y}: {v[:5]} -> {nv[:5]}  degrau {g:+.2f} US$ ({f:+.2%})")

## Parte 2: o que cada série preserva

In [ ]:
var_real, ret_real = carteira(px, vigente)
mostra("\nParte 2. O que cada série preserva (jan/2025 a set/2026)")
mostra(f"   ganho real de quem rolou 1 contrato: {var_real.sum():+.2f} US$/barril; retorno composto {pc((1 + ret_real).prod() - 1)}")
for rot, s in (("sem ajuste", sem), ("diferença", dif), ("razão", raz)):
    mostra(f"   {rot:10s}: variação {s.iloc[-1] - s.iloc[0]:+.2f} US$ | retorno {pc(s.iloc[-1] / s.iloc[0] - 1):>6s} |"
           f" preço em 02/01/2025 {s.iloc[0]:.2f} | máxima {s.max():.2f} em {s.idxmax():%d/%m/%Y}")
mostra(f"   preço que estava na tela em 02/01/2025 ({CONTRATOS[0][:5]}): {px.iloc[0][CONTRATOS[0]]:.2f}")
mostra(f"   erro de ganho da série sem ajuste: {pc((sem.iloc[-1] - sem.iloc[0]) / var_real.sum() - 1, 0)}")
mostra(f"   maior erro diário em %: diferença {pc((dif.pct_change().dropna() - ret_real).abs().max(), 3)}"
       f" | razão {pc((raz.pct_change().dropna() - ret_real).abs().max(), 4)}")
mostra(f"   maior erro diário em pontos: diferença {(dif.diff().dropna() - var_real).abs().max():.4f}"
       f" | razão {(raz.diff().dropna() - var_real).abs().max():.3f} US$")

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 3.5))
plt.plot(sem, color="gray", label="sem ajuste")
plt.plot(raz, color="black", label="ajuste por razão")
for dt in trocas:
    plt.axvline(dt, ls=":", color="black", lw=0.5)
plt.legend()
plt.show()

## Parte 3: rompimentos no dia da troca (petróleo)

In [ ]:
def rompimentos(s, n=20):
    return (s > s.shift(1).rolling(n).max()) | (s < s.shift(1).rolling(n).min())


mostra("\nParte 3. Petróleo: rompimentos de 20 dias (canal de Donchian no fechamento) no dia da troca")
for rot, s in (("sem ajuste", sem), ("diferença", dif), ("razão", raz)):
    r = rompimentos(s)
    mostra(f"   {rot:10s}: {int(r.sum())} rompimentos no período; {int(r.loc[trocas].sum())} no dia de uma troca")
real_dia = ret_real.reindex(trocas)
mostra(f"   no dia da troca, a série sem ajuste variou de {pc(sem.pct_change().loc[trocas].min())} a"
       f" {pc(sem.pct_change().loc[trocas].max())}; o contrato que o trader tinha, de {pc(real_dia.min())} a"
       f" {pc(real_dia.max())}")

## Parte 4: o minicontrato de Ibovespa pelo preço justo

In [ ]:
S = ibov["Close"].dropna()
taxa = cdi.reindex(S.index).ffill().bfill()                     # CDI do dia, em % ao dia
dias = S.index


def vencimento(ano, mes):
    """Quarta-feira mais próxima do dia 15 (regra da B3 para o Ibovespa futuro); se não houver pregão, o seguinte."""
    d15 = pd.Timestamp(ano, mes, 15)
    quarta = d15 + pd.Timedelta(days=(2 - d15.weekday() + 3) % 7 - 3)
    pos = dias.searchsorted(quarta)
    return dias[pos] if pos < len(dias) else None


VENC = [v for v in (vencimento(a, m) for a in range(2005, 2027) for m in (2, 4, 6, 8, 10, 12)) if v is not None]
VENC = [v for v in VENC if v > dias[0]]
VENC.append(pd.Timestamp("2026-10-14"))                         # vencimento seguinte (outubro), para o fim da série
pos_venc = [dias.searchsorted(v) for v in VENC]                 # posição do vencimento entre os pregões
n_dias = len(dias)


def du(i, k):
    """Pregões do dia i até o vencimento k (no fim da série, aproxima os pregões que ainda não existem)."""
    if pos_venc[k] < n_dias:
        return max(pos_venc[k] - i, 0)
    return n_dias - 1 - i + int(np.busday_count(dias[-1].date(), VENC[k].date()))


# contratos: um por vencimento; preço justo = Ibovespa x (1 + CDI do dia) ^ pregões até o vencimento
# (o Ibovespa é índice de retorno total: os proventos já estão nele, e a conta não desconta dividendos)
fut = pd.DataFrame(index=dias, columns=[f"V{k}" for k in range(len(VENC))], dtype=float)
for k in range(len(VENC)):
    ini_k = 0 if k == 0 else pos_venc[k - 1]                     # o contrato entra na série no vencimento anterior
    fim_k = min(pos_venc[k], n_dias - 1)
    for i in range(ini_k, fim_k + 1):
        fut.iat[i, k] = S.iat[i] * (1 + taxa.iat[i]) ** du(i, k)
# série do gráfico: troca no dia do vencimento (o contrato que vence sai; o seguinte entra)
vig_w = pd.Series("V0", index=dias)
for k in range(1, len(VENC)):
    if pos_venc[k - 1] < n_dias:
        vig_w.iloc[pos_venc[k - 1]:] = f"V{k}"
trocas_w = [dias[p] for p in pos_venc[:-1] if 0 < p < n_dias]
sem_w, dif_w, raz_w, saltos_w = emenda(fut, vig_w, trocas_w)
_, ret_w = carteira(fut, vig_w)
deg = pd.Series([f for *_, f in saltos_w], index=trocas_w)
anos = (dias[-1] - dias[0]).days / 365.25
cdi_acum = (1 + taxa.iloc[1:]).prod() - 1
mostra(f"\nParte 4. Minicontrato de Ibovespa pelo preço justo, {dias[0]:%m/%Y} a {dias[-1]:%m/%Y}: {len(trocas_w)} trocas")
mostra(f"   degrau por troca: mediana {pc(deg.median(), 2)}, de {pc(deg.min(), 2)} a {pc(deg.max(), 2)};"
       f" soma dos degraus {pc(deg.sum(), 0)}")
for rot, tot in (("Ibovespa à vista", S.iloc[-1] / S.iloc[0] - 1), ("futuro sem ajuste (o gráfico)", sem_w.iloc[-1] / sem_w.iloc[0] - 1),
                 ("quem ficou comprado e rolou", (1 + ret_w).prod() - 1), ("ajuste por razão", raz_w.iloc[-1] / raz_w.iloc[0] - 1),
                 ("CDI no período", cdi_acum)):
    mostra(f"   {rot:30s}: {pc(tot, 0):>6s} no total; {pc((1 + tot) ** (1 / anos) - 1):>6s} ao ano")
r_sem, r_raz = rompimentos(sem_w), rompimentos(raz_w)
falsos = r_sem.loc[trocas_w] & ~r_raz.loc[trocas_w]
mostra(f"   rompimentos de 20 dias no dia da troca: sem ajuste {int(r_sem.loc[trocas_w].sum())} de {len(trocas_w)} trocas;"
       f" ajustada {int(r_raz.loc[trocas_w].sum())}; criados só pelo degrau: {int(falsos.sum())}")
mostra(f"   num dia qualquer, a série sem ajuste rompe o canal de 20 dias em {pc(r_sem.iloc[20:].mean(), 0)} dos pregões")
mostra(f"   rompimentos de alta no dia da troca, sem ajuste: {int((sem_w.loc[trocas_w] > sem_w.shift(1).rolling(20).max().loc[trocas_w]).sum())}")
SERIES_W = pd.DataFrame({"sem": sem_w, "razao": raz_w, "vista": S})   # usado na figura

In [ ]:
import matplotlib.pyplot as plt
b = SERIES_W.iloc[0]
plt.figure(figsize=(10, 3.5))
plt.semilogy(SERIES_W["sem"] / b["sem"] * 100, color="black", label="futuro sem ajuste (o gráfico)")
plt.semilogy(SERIES_W["razao"] / b["razao"] * 100, color="gray", label="ajuste por razão (o bolso)")
plt.legend()
plt.show()

## Parte 5: o degrau esperado por nível de juro

In [ ]:
mostra("\nParte 5. Degrau esperado na troca do minicontrato de Ibovespa (vencimentos a cada 2 meses, cerca de 42 pregões)")
for juro in (0.10, 0.125, 0.15):
    mostra(f"   juro de {pc(juro)} ao ano: o contrato seguinte vale cerca de {pc((1 + juro) ** (42 / 252) - 1, 2)} a mais"
           " (o Ibovespa reinveste os proventos, por isso a conta não desconta dividendos)")

## Parte 6: o açúcar da 3ª edição, refeito (exemplo extra do hub)

A 3ª edição mostrava o contrato perpétuo de açúcar numa tela de plataforma. Aqui ele é refeito com cinco contratos de açúcar nº 11 da ICE, pelos três métodos da Parte 1. As trocas são didáticas, como no petróleo.

In [ ]:
# Extra do hub: o açúcar do Gráfico A2.1 da 3ª edição, refeito pelos três métodos (mesmas funções da Parte 1)
ACUCAR = ["SBH27.NYB", "SBK27.NYB", "SBN27.NYB", "SBV27.NYB", "SBH28.NYB"]     # açúcar nº 11 (ICE), em centavos de US$ por libra-peso
CACHE_ACUCAR = "dados_anexo2_acucar.pkl"
if os.path.exists(CACHE_ACUCAR):
    pa = pd.read_pickle(CACHE_ACUCAR)
else:
    pa = pd.concat({c: baixa(c, start=INI, end=FIM, auto_adjust=False, multi_level_index=False)["Close"]
                    for c in ACUCAR}, axis=1, sort=True)
    pd.to_pickle(pa, CACHE_ACUCAR)
pa = pa.dropna()
na = len(pa)
trocas_a = [pa.index[int(na * k / len(ACUCAR))] for k in range(1, len(ACUCAR))]   # 4 dias de troca, didáticos
vig_a = pd.Series(ACUCAR[0], index=pa.index)
for k, dt in enumerate(trocas_a, start=1):
    vig_a[pa.index >= dt] = ACUCAR[k]
sem_a, dif_a, raz_a, saltos_a = emenda(pa, vig_a, trocas_a)
var_a, ret_a = carteira(pa, vig_a)
mostra(f"\nParte 6. Açúcar (extra do hub): {pa.index[0]:%d/%m/%Y} a {pa.index[-1]:%d/%m/%Y}, {len(trocas_a)} trocas didáticas")
for dt, v, nv, g, f in saltos_a:
    mostra(f"   {dt:%d/%m/%Y}: {v[:5]} -> {nv[:5]}  degrau {g:+.2f} centavos ({f:+.2%})")
mostra(f"   ganho real de quem rolou 1 contrato: {var_a.sum():+.2f} centavos por libra-peso; retorno composto {pc((1 + ret_a).prod() - 1)}")
for rot, s in (("sem ajuste", sem_a), ("diferença", dif_a), ("razão", raz_a)):
    r = rompimentos(s)
    mostra(f"   {rot:10s}: variação {s.iloc[-1] - s.iloc[0]:+.2f} | retorno {pc(s.iloc[-1] / s.iloc[0] - 1):>6s} |"
           f" primeiro preço {s.iloc[0]:.2f} | rompimentos de 20 dias no dia da troca: {int(r.loc[trocas_a].sum())}")
plt.figure(figsize=(10, 3.5))
plt.plot(sem_a, color="gray", label="sem ajuste")
plt.plot(dif_a, color="black", ls="--", label="ajuste por diferença")
plt.plot(raz_a, color="black", label="ajuste por razão")
for dt in trocas_a:
    plt.axvline(dt, ls=":", color="black", lw=0.5)
plt.legend()
plt.show()


## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `emenda` da Parte 1. Mostre por que `dif[antes] += gap` preserva os pontos e `raz[antes] *= fator` preserva as porcentagens." Confira com a Parte 2, que imprime o ganho de quem rolou 1 contrato (+7,47 US$ por barril) e a variação de cada série: +6,20 sem ajuste, +7,47 por diferença e +7,36 por razão.
2. **Adaptar:** "Troque a lista `CONTRATOS` da célula de abertura pelos contratos de [ATIVO] no Yahoo Finance, em ordem de vencimento, e dê outro nome ao arquivo `CACHE`, para não ler a cópia do petróleo. Não mude as funções `emenda`, `carteira` e `rompimentos`." Antes de trocar, rode o notebook como está e confira se a Parte 2 continua dando os números do petróleo do livro e se a Parte 4 continua dando 130 trocas, com degrau mediano de 1,78%.

Mais pedidos deste anexo (entender e aplicar ao seu gráfico) estão na página do anexo, no hub do livro.

## Para ir além
1. Na Parte 1, troque a data das trocas para 10 pregões antes. O degrau muda?
2. Na Parte 4, troque a troca do dia do vencimento para 3 pregões antes. Os rompimentos falsos somem?
3. Na Parte 4, use um canal de 55 dias no lugar do de 20. Quantas trocas ainda rompem?
4. Na Parte 5, calcule o degrau do minidólar: diferença entre o CDI e o juro americano em um mês.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.